# Machine Learning Concepts: Customer Intelligence

## Instructor-led companion notebook

This notebook isolates the machine-learning portion of the Instamart customer project: **unsupervised RFM segmentation** and **supervised lapse prediction**. It explains the concepts and runs the project's existing scripts rather than duplicating their implementation.

The data is fictional and generated locally. No API key or network access is needed.

### Before class

1. Use Python 3.11 or newer and install dependencies with `python -m pip install -r requirements.txt`.
2. Open and run this notebook from the repository root.
3. Run cells in order. The first section generates and cleans data needed by the models.

The separate LLM class?ticket triage, policy RAG, and refund-agent tasks?is in [customer_intelligence_main_task.ipynb](./customer_intelligence_main_task.ipynb).

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    candidates = [ROOT / "instamart-customer-copilot", ROOT.parent]
    ROOT = next((candidate for candidate in candidates if (candidate / "generate_data.py").exists()), ROOT)
if not (ROOT / "generate_data.py").exists():
    raise FileNotFoundError("Open this notebook from the repository root.")

print("Project root:", ROOT)
print("Top-level files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

def run_script(relative_path):
    """Run a repository script with the current notebook interpreter."""
    subprocess.run([sys.executable, str(ROOT / relative_path)], cwd=ROOT, check=True)

## 1. Prepare clean data for modeling

Models should use validated, trusted inputs rather than raw exports. This setup regenerates the synthetic source data, applies the project's cleaning rules, and checks the expected data-quality checkpoints before modeling.

In [ ]:
run_script("generate_data.py")
run_script("src/part_a_clean.py")
run_script("src/part_a_validate.py")

import pandas as pd

clean_dir = ROOT / "data" / "clean"
customers = pd.read_csv(clean_dir / "customers.csv")
orders = pd.read_csv(clean_dir / "orders.csv", parse_dates=["order_ts"])
print("Customers:", customers.shape, "Orders:", orders.shape)
display(customers.head(3))
display(orders.head(3))

## Part B.1 — Unsupervised learning: RFM customer segmentation

### Concepts

- **RFM features** summarize customer behavior: *recency* is days since the last completed order, *frequency* is the number of completed orders, and *monetary* is completed-order spend.
- **Standardization** puts features on comparable scales. Without it, monetary values can dominate distance calculations simply because their numeric range is larger.
- **K-means inertia** is the sum of squared distances from each customer to their assigned cluster center. Lower inertia means tighter clusters, but it always decreases as K increases, so use the elbow as a heuristic rather than a standalone selection rule.
- **Silhouette score** measures how well-separated clusters are compared with their internal cohesion. Higher values are generally better; scores near zero indicate overlapping clusters and negative values can indicate poor assignments.
- **PCA** projects features into fewer dimensions for inspection. It can help visualize structure but does not itself establish that clusters are useful.

The project computes RFM from completed orders at a fixed `2025-10-01` snapshot, standardizes the features, calculates inertia and silhouette for K=2 through K=8, then fits K=4 and names clusters from their profiles. The sweep is displayed below and saved to `outputs/segment_k_sweep.csv`.

In [ ]:
run_script("src/part_b_segments.py")
k_sweep = pd.read_csv(ROOT / "outputs" / "segment_k_sweep.csv")
print("Part B: K-means inertia and silhouette by cluster count")
display(k_sweep.round({"inertia": 1, "silhouette": 3}))
segments = pd.read_csv(ROOT / "outputs" / "customer_segments.csv")
print("Segmented customers:", len(segments))
display(segments.head())
display(segments.groupby("segment_name").agg(
    customers=("customer_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
).round(2).sort_values("avg_monetary", ascending=False))

### Discuss: segmentation

1. Why should recency, frequency, and monetary features be scaled before K-means?
2. What information is lost when the three RFM dimensions are projected with PCA?
3. How would you test whether the segments are useful for a business decision, rather than merely mathematically separated?

## Part B.2 — Supervised learning: predicting customer lapse

### Concepts

- **Supervised classification** learns a mapping from historical features to a labeled outcome. Here the label marks customers who do not place a completed order in September.
- **Temporal cutoffs prevent leakage.** Features are built only from history before `2025-09-01`; September is used to define the outcome, not to construct predictors.
- **Train/test separation and pipelines** help estimate generalization. Imputation, scaling, and encoding must be fitted on training data only; fitting preprocessing on all rows can leak information into evaluation.
- **Class imbalance** can make accuracy misleading. Precision, recall, F1, and the confusion matrix reveal different error trade-offs. Training-only oversampling is compared with the baseline.

The script compares logistic regression and random forest, performs a stratified split and cross-validation, reports test metrics, saves the selected model, and builds an illustrative retention shortlist. The Premium-only shortlist uses whole-window RFM and is a campaign-prioritization example, not a leakage-free evaluation.

In [ ]:
run_script("src/part_b_lapse.py")
retention_path = ROOT / "outputs" / "retention_list.csv"
retention = pd.read_csv(retention_path)
print("Retention-list rows:", len(retention))
display(retention.head(10))
print("Saved model:", (ROOT / "outputs" / "lapse_model.joblib").exists())

### Discuss: leakage and evaluation

1. Which dates can enter the lapse-model features, and what is September used for?
2. Why should scaling and encoding be fitted only on training data?
3. Why is the Premium-only retention list a campaign-prioritization example rather than a leakage-free evaluation?
4. Which metric would you prioritize if missing a likely lapse is more costly than contacting a customer who would have returned anyway?